In [13]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
import xgboost as xgb
import optuna

print("Libraries imported successfully.")

# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)

# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 50 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm', 'view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()

# --- Prepare Target Variables ---
y_true = df_train['sale_price'].copy() # Original dollar value
y_log = np.log1p(y_true) # Log-transformed value for mean model training
print("Setup complete.")

Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [14]:
# =============================================================================
# BLOCK 2: MASTER FEATURE ENGINEERING
# =============================================================================
print("--- Starting Master Feature Engineering ---")

def create_master_features(df_train, df_test):
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

    # Date and Age Features
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['age'] = all_data['sale_year'] - all_data['year_built']
    all_data['age'] = all_data['age'].apply(lambda x: max(x, 0))

    # Brute-Force Interactions
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # Advanced Ratio and Interaction Features
    epsilon = 1e-6
    all_data['imp_val_per_sqft'] = all_data['imp_val'] / (all_data['sqft'] + epsilon)
    all_data['lot_to_house_ratio'] = all_data['sqft_lot'] / (all_data['sqft'] + epsilon)
    all_data['grade_x_sqft'] = all_data['grade'] * all_data['sqft']
    all_data['grade_x_age'] = all_data['grade'] * all_data['age']

    # Text Features
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf.fit_transform(all_data[col]))
        all_data = pd.concat([all_data, pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])], axis=1)

    # Final Cleanup
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket', 'sale_year']
    existing_cols_to_drop = [col for col in cols_to_drop if col in all_data.columns]
    all_data = all_data.drop(columns=existing_cols_to_drop)
    all_data.fillna(0, inplace=True)

    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train', 'sale_price', 'sale_price_log'], errors='ignore')
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train', 'sale_price', 'sale_price_log'], errors='ignore')
    
    return X, X_test, test_ids

X, X_test, test_ids = create_master_features(df_train, df_test)
print(f"\nMaster FE complete. Total features: {X.shape[1]}")
gc.collect()

--- Starting Master Feature Engineering ---

Master FE complete. Total features: 113


1556

In [15]:
# =============================================================================
# BLOCK 3: TUNE AND TRAIN THE MEAN PREDICTION MODEL (LOG TARGET)
# =============================================================================
print("\n--- STAGE 1, PART 1: Tuning Mean Prediction Model ---")

def objective_mean(trial):
    train_x, val_x, train_y, val_y = train_test_split(X, y_log, test_size=0.2, random_state=RANDOM_STATE)
    dtrain = xgb.DMatrix(train_x, label=train_y)
    dval = xgb.DMatrix(val_x, label=val_y)
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist',
        'eta': trial.suggest_float('eta', 0.02, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 7, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'lambda': trial.suggest_float('lambda', 1e-4, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-4, 1.0, log=True),
    }
    model = xgb.train(params, dtrain, num_boost_round=2500, evals=[(dval, 'eval')], early_stopping_rounds=100, verbose_eval=False)
    preds = model.predict(dval, iteration_range=(0, model.best_iteration))
    return np.sqrt(mean_squared_error(val_y, preds))

study_mean = optuna.create_study(direction='minimize')
study_mean.optimize(objective_mean, n_trials=N_OPTUNA_TRIALS)
best_params_mean = study_mean.best_params
print(f"\n# Mean Model Tuning Complete. Best Validation (Log) RMSE: {study_mean.best_value:.4f}")

# --- STAGE 1, PART 2: K-Fold Training of Mean Model ---
print("\n# STAGE 1, PART 2: K-Fold Training of Mean Model...")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_mean_preds = np.zeros(len(X))
test_mean_preds = np.zeros(len(X_test))
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']
final_params_mean = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', **best_params_mean}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Mean Model - Fold {fold+1}/{N_SPLITS}...")
    dtrain = xgb.DMatrix(X.iloc[train_idx], label=y_log.iloc[train_idx])
    dval = xgb.DMatrix(X.iloc[val_idx], label=y_log.iloc[val_idx])
    model = xgb.train(final_params_mean, dtrain, num_boost_round=2500, evals=[(dval, 'eval')], early_stopping_rounds=100, verbose_eval=False)
    
    val_preds_log = model.predict(dval, iteration_range=(0, model.best_iteration))
    test_preds_log = model.predict(xgb.DMatrix(X_test), iteration_range=(0, model.best_iteration))
    
    oof_mean_preds[val_idx] = np.expm1(val_preds_log)
    test_mean_preds += np.expm1(test_preds_log) / N_SPLITS

final_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
print(f"\n# Mean model K-Fold training complete. Final OOF RMSE: ${final_mean_rmse:,.2f}")
print("-" * 50)

[I 2025-07-09 18:01:22,645] A new study created in memory with name: no-name-ef7c9969-1f26-4a35-aa77-1cbf93508516



--- STAGE 1, PART 1: Tuning Mean Prediction Model ---


[I 2025-07-09 18:02:07,595] Trial 0 finished with value: 0.16422530765571475 and parameters: {'eta': 0.0390617780253073, 'max_depth': 11, 'subsample': 0.8289343938345655, 'colsample_bytree': 0.9308318703565877, 'min_child_weight': 7, 'lambda': 0.1528808048939695, 'alpha': 0.00013764964135937747}. Best is trial 0 with value: 0.16422530765571475.
[I 2025-07-09 18:02:40,091] Trial 1 finished with value: 0.16218041347332715 and parameters: {'eta': 0.04421742178223089, 'max_depth': 9, 'subsample': 0.6967122640912089, 'colsample_bytree': 0.8425159758917634, 'min_child_weight': 2, 'lambda': 0.004934343910678847, 'alpha': 0.022016841383839157}. Best is trial 1 with value: 0.16218041347332715.
[I 2025-07-09 18:03:20,861] Trial 2 finished with value: 0.1654368315787774 and parameters: {'eta': 0.04047164263635208, 'max_depth': 12, 'subsample': 0.94393688924796, 'colsample_bytree': 0.8491979904765199, 'min_child_weight': 10, 'lambda': 0.007841539514808314, 'alpha': 0.0011696430651268715}. Best is 


# Mean Model Tuning Complete. Best Validation (Log) RMSE: 0.1565

# STAGE 1, PART 2: K-Fold Training of Mean Model...
  Mean Model - Fold 1/5...
  Mean Model - Fold 2/5...
  Mean Model - Fold 3/5...
  Mean Model - Fold 4/5...
  Mean Model - Fold 5/5...

# Mean model K-Fold training complete. Final OOF RMSE: $117,582.43
--------------------------------------------------


In [16]:
# =============================================================================
# BLOCK 4: TUNE AND TRAIN THE ERROR PREDICTION MODEL (DOLLAR-SPACE)
# =============================================================================
print("\n--- STAGE 2, PART 1: Tuning Error Prediction Model ---")

# Define the new target: the absolute error in the ORIGINAL DOLLAR-SPACE
error_target_dollar = np.abs(y_true - oof_mean_preds)

# Add the mean model's predictions as a feature
X_for_error = X.copy()
X_for_error['mean_pred_oof'] = oof_mean_preds
X_test_for_error = X_test.copy()
X_test_for_error['mean_pred_oof'] = test_mean_preds

def objective_error(trial):
    train_x, val_x, train_y, val_y = train_test_split(X_for_error, error_target_dollar, test_size=0.2, random_state=RANDOM_STATE)
    dtrain = xgb.DMatrix(train_x, label=train_y)
    dval = xgb.DMatrix(val_x, label=val_y)
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist',
        'eta': trial.suggest_float('eta', 0.01, 0.05),
        'max_depth': trial.suggest_int('max_depth', 5, 9),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = xgb.train(params, dtrain, num_boost_round=1500, evals=[(dval, 'eval')], early_stopping_rounds=50, verbose_eval=False)
    preds = model.predict(dval, iteration_range=(0, model.best_iteration))
    return np.sqrt(mean_squared_error(val_y, preds))

study_error = optuna.create_study(direction='minimize')
study_error.optimize(objective_error, n_trials=N_OPTUNA_TRIALS)
best_params_error = study_error.best_params
print(f"\n# Error Model Tuning Complete. Best Validation RMSE: ${study_error.best_value:,.2f}")

# --- STAGE 2, PART 2: K-Fold Training of Error Model ---
print("\n# STAGE 2, PART 2: K-Fold Training of Error Model...")
oof_error_preds = np.zeros(len(X))
test_error_preds = np.zeros(len(X_test))
final_params_error = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', **best_params_error}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error, grade_for_stratify)):
    print(f"  Error Model - Fold {fold+1}/{N_SPLITS}...")
    dtrain = xgb.DMatrix(X_for_error.iloc[train_idx], label=error_target_dollar.iloc[train_idx])
    dval = xgb.DMatrix(X_for_error.iloc[val_idx], label=error_target_dollar.iloc[val_idx])
    model = xgb.train(final_params_error, dtrain, num_boost_round=2000, evals=[(dval, 'eval')], early_stopping_rounds=100, verbose_eval=False)
    oof_error_preds[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration))
    test_error_preds += model.predict(xgb.DMatrix(X_test_for_error), iteration_range=(0, model.best_iteration)) / N_SPLITS

final_error_rmse = np.sqrt(mean_squared_error(error_target_dollar, oof_error_preds))
print(f"\n# Error model K-Fold training complete. Final OOF RMSE: ${final_error_rmse:,.2f}")
print("-" * 50)


--- STAGE 2, PART 1: Tuning Error Prediction Model ---


[I 2025-07-09 18:30:20,075] A new study created in memory with name: no-name-c4479731-fe07-4100-9d86-646d76ea1ec1
[I 2025-07-09 18:30:46,251] Trial 0 finished with value: 76160.69899987763 and parameters: {'eta': 0.015655029117498977, 'max_depth': 9, 'subsample': 0.7750202424836092, 'colsample_bytree': 0.7368147335173387, 'min_child_weight': 7}. Best is trial 0 with value: 76160.69899987763.
[I 2025-07-09 18:31:00,385] Trial 1 finished with value: 76628.11055628004 and parameters: {'eta': 0.04902718314805775, 'max_depth': 5, 'subsample': 0.9766256189252905, 'colsample_bytree': 0.8084223459879442, 'min_child_weight': 8}. Best is trial 0 with value: 76160.69899987763.
[I 2025-07-09 18:31:04,496] Trial 2 finished with value: 76417.80280397057 and parameters: {'eta': 0.04346261030973923, 'max_depth': 7, 'subsample': 0.8154875321761893, 'colsample_bytree': 0.964193306935267, 'min_child_weight': 10}. Best is trial 0 with value: 76160.69899987763.
[I 2025-07-09 18:31:10,577] Trial 3 finished 


# Error Model Tuning Complete. Best Validation RMSE: $76,114.34

# STAGE 2, PART 2: K-Fold Training of Error Model...
  Error Model - Fold 1/5...
  Error Model - Fold 2/5...
  Error Model - Fold 3/5...
  Error Model - Fold 4/5...
  Error Model - Fold 5/5...

# Error model K-Fold training complete. Final OOF RMSE: $76,419.92
--------------------------------------------------


In [17]:
# =============================================================================
# BLOCK 5: FINAL ASYMMETRIC CALIBRATION AND SUBMISSION
# =============================================================================
print("\n--- Final Asymmetric Calibration ---")

oof_error_final = np.clip(oof_error_preds, 0, None)
best_a, best_b, best_metric = 1.0, 1.0, float('inf')

# Wider search for multipliers as the error dynamic has changed
for a in np.arange(1.5, 2.5, 0.02):
    for b in np.arange(1.5, 2.5, 0.02):
        lower = oof_mean_preds - oof_error_final * a
        upper = oof_mean_preds + oof_error_final * b
        metric, coverage = winkler_score(y_true, lower, upper, alpha=COMPETITION_ALPHA, return_coverage=True)
        if coverage > 0.85 and metric < best_metric: # Ensure reasonable coverage
            best_metric = metric
            best_a, best_b = a, b
            print(f"New Best! a={best_a:.2f}, b={best_b:.2f}, Score={best_metric:,.2f}, Cov={coverage:.2%}")

print(f"\nGrid search complete. Final OOF Score: {best_metric:,.2f}. Best multipliers: a={best_a:.2f}, b={best_b:.2f}")

# --- Create Final Submission ---
print("\nCreating final submission file...")
test_error_final = np.clip(test_error_preds, 0, None)
final_lower = test_mean_preds - test_error_final * best_a
final_upper = test_mean_preds + test_error_final * best_b
final_upper = np.maximum(final_lower, final_upper)

submission_df = pd.DataFrame({'id': test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
submission_df.to_csv('submission_v2_advanced.csv', index=False)
print("\n'submission_v2_advanced.csv' created successfully!")
display(submission_df.head())


--- Final Asymmetric Calibration ---
New Best! a=1.50, b=2.12, Score=372,259.37, Cov=85.13%
New Best! a=1.50, b=2.14, Score=371,836.62, Cov=85.28%
New Best! a=1.50, b=2.16, Score=371,459.14, Cov=85.43%
New Best! a=1.50, b=2.18, Score=371,122.56, Cov=85.58%
New Best! a=1.50, b=2.20, Score=370,828.07, Cov=85.73%
New Best! a=1.50, b=2.22, Score=370,571.65, Cov=85.86%
New Best! a=1.50, b=2.24, Score=370,350.35, Cov=85.99%
New Best! a=1.50, b=2.26, Score=370,161.13, Cov=86.11%
New Best! a=1.50, b=2.28, Score=370,000.70, Cov=86.23%
New Best! a=1.50, b=2.30, Score=369,869.35, Cov=86.33%
New Best! a=1.50, b=2.32, Score=369,769.04, Cov=86.45%
New Best! a=1.50, b=2.34, Score=369,698.09, Cov=86.56%
New Best! a=1.50, b=2.36, Score=369,658.03, Cov=86.68%
New Best! a=1.50, b=2.38, Score=369,652.64, Cov=86.81%
New Best! a=1.52, b=2.24, Score=369,566.36, Cov=86.25%
New Best! a=1.52, b=2.26, Score=369,377.14, Cov=86.37%
New Best! a=1.52, b=2.28, Score=369,216.71, Cov=86.49%
New Best! a=1.52, b=2.30, S

,id,pi_lower,pi_upper
0,0,779557.758477,1.075994e+06
1,1,469310.332363,8.079540e+05
2,2,431815.354805,7.004055e+05
3,3,307021.900781,4.491108e+05
4,4,327509.331289,9.714454e+05
